In [ ]:
"""
code to post process VNA measurments. expects .npz files with data made with following structure: 


data_matrices = {
        'positions [cm]': real_positions,
        'freq [MHz]': np.zeros(NUM_FREQ_POINTS),
        'S11': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),
        'S21': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),
        'S31': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),
        'S41': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex)
    } 


The VNA ports are as follows: 

Port 1: Signal port; sending out signal with power P and frequency F to TWA port 1 
Port 2: return from TWA port 2 
Port 3: return from TWA port 3
Port 4: return from the difference port of the Eprobe's 180 degree hybrid difference port  

"""

"\ncode to post process VNA measurments. expects .npz files with data made with following structure: \n\n\ndata_matrices = {\n        'S11': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),\n        'S21': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),\n        'S31': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex),\n        'S41': np.zeros((num_measurments, NUM_FREQ_POINTS), dtype=complex)\n    }\n\nThe VNA ports are as follows: \n\nPort 1: Signal port; sending out signal with power P and frequency F to TWA port 1 \nPort 2: return from TWA port 2 \nPort 3: return from TWA port 3\nPort 4: return from the difference port of the Eprobe's 180 degree hybrid difference port  \n\n"

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 1. USER INPUTS
# ==========================================
FILENAME = 'vna_pos_data_1.npz'

# The specific frequency you want to extract the spatial wave from (in MHz)
TARGET_FFT_FREQ_MHZ = 96.0  

clight = 3e8

In [ ]:
# ==========================================
# 2. DYNAMIC DATA UNPACKING
# ==========================================
print(f"Loading {FILENAME}...")
data = np.load(FILENAME)

# Unpack the axes directly from the file!
freqs_mhz = data['freq [MHz]']
positions_cm = data['positions [cm]']

# Unpack the complex S-parameter matrices
S11 = data['S11']
S21 = data['S21']
S31 = data['S31']
S41 = data['S41']

num_measurements = len(positions_cm)

# Dynamically calculate the physical dx step size used in the experiment
dx_real_cm = positions_cm[1] - positions_cm[0]
dx_real_meters = dx_real_cm / 100.0
print(f"Detected spatial step size: {dx_real_cm:.4f} cm")

# Find the column index closest to our target 96 MHz
freq_idx = np.argmin(np.abs(freqs_mhz - TARGET_FFT_FREQ_MHZ))
actual_freq_found = freqs_mhz[freq_idx]
print(f"Extracting spatial data for FFT at {actual_freq_found:.2f} MHz")

In [ ]:
# ==========================================
# 3. S11, S21, S31 MAGNITUDE PROCESSING (dB)
# ==========================================
# Plot S11 from the very first cart position (Index 0). assumes cart has little impact on S11. 
# Formula: dB = 20 * log10(|S|)
s11_mag_linear = np.abs(S11[0, :])
s11_db = 20 * np.log10(s11_mag_linear)

s21_mag_linear = np.abs(S21[0, :])
s21_db = 20 * np.log10(s21_mag_linear)

s31_mag_linear = np.abs(S31[0, :])
s31_db = 20 * np.log10(s31_mag_linear)

In [ ]:
# ==========================================
# PLOTTING Sij
# ==========================================
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
# --- Plot 1: S11, s21, S31 vs Frequency ---
ax.plot(freqs_mhz, s11_db, color='black', marker='.', linewidth=2, label='S11')
ax.plot(freqs_mhz, s21_db, color='red', marker='.', linewidth=2, label='S21')
ax.plot(freqs_mhz, s31_db, color='blue', marker='.', linewidth=2, linestyle='--', label='S31')

ax.set_title(r'$S$ Magnitude (Cart Position 0)', fontsize=14)
ax.set_xlabel('Frequency (MHz)', fontsize=12)
ax.set_ylabel('Magnitude (dB)', fontsize=12)
ax.axvline(TARGET_FFT_FREQ_MHZ, color='red', linestyle='--', label=f'Target: {TARGET_FFT_FREQ_MHZ:.1f} MHz')
ax.grid(True, which='both', linestyle='--', alpha=0.7)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 4. S41 SPATIAL FFT PROCESSING (k-space)
# ==========================================

# Extract the 1D spatial array for just that frequency column
s41_spatial_wave = S41[:, freq_idx]

# Perform the Spatial FFT
s41_fft = np.fft.fftshift(np.fft.fft(s41_spatial_wave))
s41_fft_mag = np.abs(s41_fft)

# Calculate the wavenumber (k-space) x-axis in rad/m using the dynamic dx
k_axis = np.fft.fftshift(np.fft.fftfreq(num_measurements, d=dx_real_meters)) * 2 * np.pi

n_axis = k_axis * clight / (2 * np.pi * actual_freq_found * 1e6)

In [ ]:
# ==========================================
# PLOTTING FFT
# ==========================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.plot(k_axis, s41_fft_mag, marker='o', color='purple', linewidth=2)
ax1.set_title(rf'$S_{41}$ Spatial Spectrum at {actual_freq_found:.2f} MHz ($k$-space)', fontsize=14)
ax1.set_xlabel('Wavenumber $k$ (rad/m)', fontsize=12)
ax1.set_ylabel('FFT Magnitude', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(n_axis, s41_fft_mag, marker='o', color='red', linewidth=2)
ax2.set_title(rf'$S_{41}$ Spatial Spectrum at {actual_freq_found:.2f} MHz ($n$-space)', fontsize=14)
ax2.set_xlabel(r'n_{||}', fontsize=12)
ax2.set_ylabel('FFT Magnitude', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()